# モジュール 5: 論文 Table 2 と答え合わせする ★講習会の山場

**所要時間 30 分**

**このノートで身につくこと**

1. 自分が出した 22 輝線の強度を、論文 Table 2 と**1 行ずつ**突き合わせる
2. **「箱の選び方」を数値で診断する**（ratio の温度依存の傾き）
3. 合わない線について、**原因を切り分ける手順**を身につける
4. ★ **独立な 2 つ目の測定を持ってくる**のが最強の切り分けだと理解する

前提: モジュール 2（22 輝線のフィット）、モジュール 4（箱の選択）。

---

**なぜこの天体なのか**: 論文 Table 2 は region 7（2011-07-02 03:07:12,
NOAA 1243）の観測強度を 23 行そのまま載せている。
**論文に数値表が載っている唯一の天体**なので、
「自分の解析が合っているか」が客観的に確かめられる。

In [ ]:
!pip install -q eispac

In [ ]:
import os
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import eispac

EIS_FILE = "data/eis/eis_20110702_030712.data.h5"
BOX = dict(y0=244, y1=274, x0=32, x1=40)          # モジュール 4 で選んだ箱


def ensure(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    os.makedirs(os.path.dirname(path), exist_ok=True)
    urllib.request.urlretrieve(url, path)
    return path


for ext in ("data", "head"):
    ensure(f"https://eis.nrl.navy.mil/level1/hdf5/2011/07/02/"
           f"eis_20110702_030712.{ext}.h5",
           f"data/eis/eis_20110702_030712.{ext}.h5")

import sys
sys.path.insert(0, "scripts")                      # 教材リポジトリの中
from lines_warren2012 import LINES, pick_component
from fit_box_spectra import average_spectrum       # モジュール 2 と同じ中身
print(f"論文 Table 2 の輝線数: {len(LINES)}")

## 5-1. 22 輝線をフィットして比べる

In [ ]:
# おおまかな形成温度（log T）。ratio の温度依存を見るために使う
LOGT = {"Si VII": 5.8, "Fe IX": 5.9, "Fe X": 6.05, "Fe XI": 6.15, "S X": 6.15,
        "Si X": 6.15, "Fe XII": 6.2, "Fe XIII": 6.25, "Fe XIV": 6.3,
        "Fe XV": 6.35, "S XIII": 6.4, "Fe XVI": 6.45, "Ar XIV": 6.5,
        "Ca XIV": 6.55, "Ca XV": 6.65, "Ca XVI": 6.7, "Ca XVII": 6.75}


def fit_box(box):
    rows = []
    for ion, wvl, tname, i_paper, sig_paper in LINES:
        w, I, s, npix = average_spectrum(EIS_FILE, wvl, **box)
        t = eispac.read_template(eispac.data.get_fit_template_filepath(tname))
        comp, ids = pick_component(t, wvl)
        f = eispac.fit_spectra(I, t, wave=w, errs=s, ncpu=1, ignore_warnings=True)
        i_fit = float(np.atleast_1d(f.fit["int"][..., comp]).ravel()[0])
        rows.append(dict(ion=ion, wvl=wvl, i_fit=i_fit, i_paper=i_paper,
                         sig_paper=sig_paper, ratio=i_fit / i_paper,
                         logt=LOGT[ion]))
    return rows


rows = fit_box(BOX)

print(f"{'line':<16}{'logT':>5}{'I_fit':>10}{'I_paper':>10}{'±':>8}{'ratio':>7}")
for r in sorted(rows, key=lambda r: r["logt"]):
    flag = "  ✓" if abs(r["ratio"] - 1) <= 0.15 else ""
    name = "{} {:.3f}".format(r["ion"], r["wvl"])
    print(f"{name:<16}{r['logt']:5.2f}{r['i_fit']:10.2f}{r['i_paper']:10.2f}"
          f"{r['sig_paper']:8.2f}{r['ratio']:7.2f}{flag}")

In [ ]:
use = [r for r in rows if r["ion"] != "Ca XVII"]     # Ca XVII はブレンド（後述）
ratios = np.array([r["ratio"] for r in use])
logt = np.array([r["logt"] for r in use])
n15 = int((np.abs(ratios - 1) <= 0.15).sum())

print(f"Ca XVII を除く {len(use)} 本について")
print(f"  median ratio  = {np.median(ratios):.2f}")
print(f"  ばらつき      = {np.std(np.log10(ratios)):.2f} dex")
print(f"  論文の 15% 以内 = {n15}/{len(use)} 本")
print(f"  論文自身の誤差  = ±22%")

**まず全体を見る。median が 0.89 = 論文より 11% 低い。**

これは論文の誤差 ±22% の中に収まっている。
「合っている」と言ってよい水準だが、**なぜ 11% 低いのか**は次節で切り分ける。

## 5-2. ★ 「箱の選び方」を数値で診断する

ratio を**形成温度に対して**並べるのが決め手になる。

| ratio の温度依存 | 意味 |
|---|---|
| 傾き ≈ 0（全体に一定倍率） | 論文と**同じ温度組成**の場所を見ている。ずれは較正の問題 |
| 傾き < 0（冷たい線が明るく熱い線が暗い） | **暖かいループ寄り**を選んでいる |
| 傾き > 0（熱い線が明るい） | **高温コア寄り**を選んでいる |

装置の較正は波長の関数であって温度の関数ではないから、
**温度に沿ったパターンが出たら、それは場所の違い**である。

In [ ]:
slope = np.polyfit(logt, np.log10(ratios), 1)[0]
print(f"log(ratio) の logT に対する傾き = {slope:+.2f}")

plt.figure(figsize=(7, 4.5))
for r in use:
    plt.plot(r["logt"], r["ratio"], "o", color="C0")
    if abs(r["ratio"] - 1) > 0.25:
        plt.annotate(f"{r['ion']} {r['wvl']:.1f}", (r["logt"], r["ratio"]),
                     fontsize=7, xytext=(3, 3), textcoords="offset points")
xx = np.linspace(logt.min(), logt.max(), 10)
plt.plot(xx, 10**np.polyval(np.polyfit(logt, np.log10(ratios), 1), xx),
         "-", color="C3", label=f"slope = {slope:+.2f}")
plt.axhline(1.0, color="k", lw=1)
plt.axhspan(0.85, 1.15, color="0.85", zorder=0, label="within 15%")
plt.yscale("log")
plt.xlabel("log T [K]")
plt.ylabel("I(ours) / I(Warren+2012)")
plt.title("ratio vs formation temperature")
plt.legend()
plt.tight_layout()
plt.show()

### 箱を変えて傾きを比べる（この演習が一番教育的）

**受講者に何通りか箱を選ばせて、この傾きを比べさせる**のが良い。

In [ ]:
BOXES = {
    "採用箱 (inter-moss)":  dict(y0=244, y1=274, x0=32, x1=40),
    "論文の箱サイズに近い":  dict(y0=246, y1=269, x0=30, x1=38),
    "適当に明るいところ":    dict(y0=200, y1=230, x0=20, x1=28),
}
for label, box in BOXES.items():
    rr = fit_box(box)
    u = [r for r in rr if r["ion"] != "Ca XVII"]
    ra = np.array([r["ratio"] for r in u])
    lt = np.array([r["logt"] for r in u])
    sl = np.polyfit(lt, np.log10(ra), 1)[0]
    print(f"{label:<22} median={np.median(ra):5.2f}  傾き={sl:+5.2f}  "
          f"15%以内={int((np.abs(ra-1) <= 0.15).sum())}/{len(u)}  "
          f"ばらつき={np.std(np.log10(ra)):.2f} dex")

**★ ここは必ず立ち止まって読むこと。**

「適当に明るいところ」の箱は、**median が 0.96 と一番 1 に近い**。
median だけ見ていたら「この箱が一番良い」と結論してしまう。

ところが同じ箱は

- **傾き −0.34**（冷たい線が明るく熱い線が暗い = 暖かいループを見ている）
- **15% 以内が 5/21 本しかない**（採用箱は 13/21）
- **ばらつき 0.26 dex**（採用箱の 2.6 倍）

つまり **個々の線は全然合っていないのに、上下のずれが打ち消し合って
median だけがきれいに見えている**。

→ **要約統計 1 つで判断しない。** 温度に沿って並べる、ばらつきを見る、
  本数を数える。この 3 つを揃えて初めて「合っている」と言える。

## 5-3. 合わない線を 1 つずつ議論する（ここが一番勉強になる）

### (a) Ca XVII 192.858 が 5 倍 → **原因がはっきりしている**

eispac 同梱の `ca_17_192_858.1c` は 192.700–193.200 Å を
**単一ガウシアンで塗るだけ**で、Fe XI 192.813 と O V の複合線を分離しない。
論文は Ko et al. (2009) の方法で分離している。

→ **モジュール 8** で自作テンプレートを作ると **5.08 → 0.75** になる。
  （SSW/IDL で同じことをすると 0.77。Python だけで再現できる）

### (b) Fe XIII 202.044 / 203.826 → **論文でも外れている**

後で DEM を解くと、この 2 本だけ I_obs/I_DEM が 1.3–2.8 になる。
**論文でも 1.80 / 1.82**、Warren et al. (2011) でも 1.87 / 1.90。
→ **原子データ側の既知の問題**であって、我々の解析の問題ではない。
  密度診断ペアであり、モジュール 6 で「実装間の差が最大の線」として再登場する。

### (c) Si VII 275.368 が 0.40、Fe XVI 262.984 が 0.62 → **未解決**

ここが正直に扱うべきところ。**容疑を 1 つずつ潰した記録**を次に示す。

### ★ 「Aで説明できる」と言ったら、Aの予測を検証する

**仮説 1: inter-moss 領域の自然なばらつき**

他論文の inter-moss 領域と、Fe XII 195.119 に対する比で並べると:

| | SiVII/FeXII | FeXVI/FeXII | SXIII/FeXII |
|---|---:|---:|---:|
| Tripathi+2011 inter-moss A | 0.0477 | 0.4381 | 0.4704 |
| Tripathi+2011 inter-moss B | 0.0230 | 0.2372 | 0.2916 |
| Tripathi+2011 inter-moss C | 0.0848 | 0.4763 | 0.5267 |
| Warren+2011 | 0.0319 | 0.7846 | 0.5793 |
| Warren+2012 region 7（論文） | 0.0583 | 0.5498 | 0.4029 |
| **我々** | **0.0240** | **0.3620** | **0.3078** |

実在の inter-moss 領域どうしで 3.7 倍ばらついている。
「だから正常」と言いたくなる。**しかしこれは誤り。**
Tripathi+2011 は**別の活動領域**であり、
「このラスターの中に論文と同じ組み合わせが実現できる」ことを保証しない。

**仮説が正しいなら成り立つはずの予測**:
「自然なばらつきなら、このラスター内に論文と同じ比の箱があるはず」

→ 230 箱を総当たりして検証した結果（`scripts/scan_perline.py`）:

- **「22 輝線の 15% 以内が 12 本以上」かつ「Si VII 比 ≥ 0.8」を満たす箱は 0 個**
- Si VII を合わせにいくと Si X 1.85、Fe XIV 1.67、Ar XIV 0.28、
  Ca XVI 0.15 と**高温側が壊滅する**

→ **予測は反証された。仮説 1 は棄却。**

### ★★ 決め手は AIA（EIS と完全に独立な測定）

論文 Table 2 の最終行は **AIA 94 Å の Fe XVIII = 7.20 DN/s**。
分光器 (EIS) とは別の装置・別のデータ経路・別の単位。

| 箱 | AIA Fe XVIII | 対論文 | SiVII 比 |
|---|---:|---:|---:|
| Si VII が合う箱 | 1.16 | **0.16** | 1.13 |
| 我々の採用箱 | 6.64 | **0.92** | 0.36 |
| 論文の箱（Fig.2 から実測） | 6.81 | **0.95** | 0.43 |

**Si VII が合う箱は AIA Fe XVIII が論文の 6 分の 1**。
そこは moss であって inter-moss ではない、と **EIS とは独立に**言える。

逆に AIA が論文と合う箱では、**Si VII は必ず 2.6 倍低い**。

→ **論文 Table 2 の Si VII 値は、同じ Table 2 の AIA Fe XVIII 値と整合しない。**
  ここまで絞れれば、著者に問い合わせる価値がある。

**潰した容疑（10 個）**: フィッターの実装 / eispac 固有の問題 / 箱の位置 /
打ち上げ後較正 2 種（Del Zanna 2013, Warren+2014）/ 実効面積のバージョン /
despike / 欠損値処理 / フィットの順番 / 未モデル化のブレンド /
「箱で説明できる」説。

**★ 「長波長側の感度劣化」も成立しない。** 同じ長波長チャンネルの
S X (0.94)、Si X (1.09)、Fe XIV (0.92/0.95)、Fe XV (0.87) は合っている。
**隣り合う波長**の Si X 258.375 と Fe XVI 262.984 が 1.09 と 0.62 に分かれるので、
波長の滑らかな関数である較正では説明できない。

## 5-4. ★★★ 決定的な事実: 独立な 2 台が揃って低い

| | 我々 | 論文 |
|---|---:|---:|
| EIS 22 輝線の median | **0.89** | 1.0 |
| AIA 94 Å Fe XVIII | **0.82 – 0.92**（測り方による） | 1.0 |

**分光器（EIS, erg 単位, NRL の level-1）と撮像（AIA, DN 単位, JSOC）という
完全に独立な 2 台が、揃って 1 割低い。**

→ 装置でも処理でもなく、**論文がわずかに明るい場所を測っている**と結論できる。
  差は論文自身の誤差 ±22% の中。

（AIA の値に幅があるのは、synoptic 1024²(2.4″/px) で測るか
  フルディスク level-1 (4096²) を EIS 格子に落として測るかの違い。
  15″×23″ の箱を粗い格子で測ると 10% 動く。モジュール 3 参照。）

## 5-5. この章の教訓

1. **一致したことより、合わない理由を説明できることが実力。**
   22 本中 13 本が 15% 以内、というのは結果の半分でしかない。
2. **「Aで説明できる」と言ったら、Aが正しいなら成り立つ予測を立てて検証する。**
   今回は「このラスター内に論文と同じ箱があるはず」という予測が反証された。
3. **独立な 2 つ目の測定を持ってくるのが最強の切り分け手段。**
   今回は AIA の Fe XVIII が、EIS とは全く別ルートの検証になった。
4. **未解決を未解決のまま正確に記述する。**
   「Si VII が合わない」ではなく
   「論文 Table 2 の Si VII 値は、同じ Table 2 の AIA 値と整合しない」
   まで絞る。ここまで来て初めて次の一手が決まる。

## 5-6. 演習

1. `BOXES` に自分で箱を足して、median と傾きの組を集める。
   **median が 1 に近くても傾きが大きい箱**を見つけられるか？
   見つかったら、それは何を意味するか。
2. Fe XII 195.119 で規格化した比（`I / I_FeXII195`）で論文と比べ直す。
   絶対較正の効果が落ちるので、**場所の違いだけ**が見えるはず。
3. Ca XVII を除かずに median を計算するとどうなるか。
   **1 本の外れ値が要約統計をどれだけ動かすか**を体感する。
4. 論文の誤差 ±22% を図に描き込んで、
   「何本が誤差の範囲内か」を数え直す。

## まとめ

- 22 輝線中 **13 本が論文の 15% 以内**、median 0.89、ばらつき 0.10 dex
- **ratio の温度依存の傾き**で箱の選び方を診断できる
- Ca XVII = ブレンド（解ける）、Fe XIII = 原子データ（論文も同じ）、
  Si VII / Fe XVI = **未解決**
- **独立な 2 台（EIS と AIA）が揃って 1 割低い** → 場所の違い

ここまでが「半日コース」の到達点。
以降は寄与関数（モジュール 6）と DEM（モジュール 7）に進む。